In [26]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("./Data/data_MBTI.csv")
print(df.head())
print(df.shape)

   Age  Gender  Education  Introversion Score  Sensing Score  Thinking Score  \
0   19    Male          0             9.47080       7.141434         6.03696   
1   27  Female          0             5.85392       6.160195         0.80552   
2   21  Female          0             7.08615       3.388433         2.66188   
3   28    Male          0             2.01892       4.823624         7.30625   
4   36  Female          1             9.91703       4.755080         5.31469   

   Judging Score    Interest Personality  
0       4.360278     Unknown        ENFP  
1       4.221421      Sports        ESFP  
2       5.127320     Unknown        ENFP  
3       5.986550      Others        INTP  
4       4.677213  Technology        ENFP  
(128061, 9)


In [6]:
print(df.isnull().sum())
display(df.head())

Age                   0
Gender                0
Education             0
Introversion Score    0
Sensing Score         0
Thinking Score        0
Judging Score         0
Interest              0
Personality           0
dtype: int64


,Age,Gender,Education,Introversion Score,Sensing Score,Thinking Score,Judging Score,Interest,Personality
0,19,Male,0,9.47080,7.141434,6.03696,4.360278,Unknown,ENFP
1,27,Female,0,5.85392,6.160195,0.80552,4.221421,Sports,ESFP
2,21,Female,0,7.08615,3.388433,2.66188,5.127320,Unknown,ENFP
3,28,Male,0,2.01892,4.823624,7.30625,5.986550,Others,INTP
4,36,Female,1,9.91703,4.755080,5.31469,4.677213,Technology,ENFP


In [7]:
# Creating new combined features from existing ones
df['Mind_Score']     = df['Introversion Score'] - df['Sensing Score']
df['Decision_Style'] = df['Thinking Score'] - df['Judging Score']
df['Overall_Score']  = df['Introversion Score'] + df['Thinking Score']

print(df[['Mind_Score', 'Decision_Style', 'Overall_Score']].head())

   Mind_Score  Decision_Style  Overall_Score
0    2.329366        1.676682       15.50776
1   -0.306275       -3.415901        6.65944
2    3.697717       -2.465440        9.74803
3   -2.804704        1.319700        9.32517
4    5.161950        0.637477       15.23172


In [10]:
X = df.drop(columns=['Personality'])
y = df['Personality']
print(y.nunique())
print(y.value_counts())

'We got Imbalanced data here so we are going to use Class Weight Technique to improve the imbalance so that model can get more accurate'

16
Personality
ENFP    34404
ENTP    24718
INFP    24711
INTP    17132
ESFP     4832
ENFJ     3883
ISFP     3456
ESTP     3334
INFJ     2919
ENTJ     2783
ISTP     2390
INTJ     1920
ESFJ      554
ESTJ      392
ISFJ      371
ISTJ      262
Name: count, dtype: int64


In [15]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(le.classes_)

['ENFJ' 'ENFP' 'ENTJ' 'ENTP' 'ESFJ' 'ESFP' 'ESTJ' 'ESTP' 'INFJ' 'INFP'
 'INTJ' 'INTP' 'ISFJ' 'ISFP' 'ISTJ' 'ISTP']


In [16]:
categorical_features = ['Gender', 'Interest']
numerical_features   = ['Age', 'Education', 
                        'Introversion Score', 'Sensing Score',
                        'Thinking Score', 'Judging Score',
                        'Mind_Score', 'Decision_Style', 'Overall_Score']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(),  categorical_features)
])

X_preprocessed = preprocessor.fit_transform(X)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed, y_encoded,
    test_size=0.2,
    random_state=42
)

In [22]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_encoded),
    y=y_encoded
)

# Convert to dictionary format TensorFlow needs
class_weight_dict = dict(enumerate(class_weights))

print("Class weights:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls}: {class_weights[i]:.2f}")

Class weights:
  ENFJ: 2.06
  ENFP: 0.23
  ENTJ: 2.88
  ENTP: 0.32
  ESFJ: 14.45
  ESFP: 1.66
  ESTJ: 20.42
  ESTP: 2.40
  INFJ: 2.74
  INFP: 0.32
  INTJ: 4.17
  INTP: 0.47
  ISFJ: 21.57
  ISFP: 2.32
  ISTJ: 30.55
  ISTP: 3.35


In [23]:
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),               
    tf.keras.layers.Dropout(0.3),                     
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Dense(32, activation='relu'),   
    
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           2,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 16)                  │             528 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13,552 (52.94 KB)

 Trainable params: 13,296 (51.94 KB)

 Non-trainable params: 256 (1.00 KB)

In [27]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,            
    restore_best_weights=True 
)

print("Starting Training...")
history = model.fit(
    X_train, y_train,
    epochs=50,                     
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,  
    callbacks=[early_stop],
    verbose=1
)

Starting Training...
Epoch 1/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.6190 - loss: 1.0747 - val_accuracy: 0.7816 - val_loss: 0.5265
Epoch 2/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.7411 - loss: 0.6768 - val_accuracy: 0.8379 - val_loss: 0.3803
Epoch 3/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.7832 - loss: 0.5682 - val_accuracy: 0.8454 - val_loss: 0.3589
Epoch 4/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.7930 - loss: 0.5359 - val_accuracy: 0.8138 - val_loss: 0.4506
Epoch 5/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.8039 - loss: 0.5129 - val_accuracy: 0.8257 - val_loss: 0.3971
Epoch 6/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.8065 - loss: 0.5000 - val_accuracy: 0.8534 - val_loss: 0.3279
Epoch 7/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.8128 - loss: 0.4770 - val_accuracy: 0.8498 - val_loss: 0.3468
Epoch 8/50
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accur

In [28]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=le.classes_))

Test Accuracy: 86.04%

Classification Report:
801/801 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
              precision    recall  f1-score   support

        ENFJ       0.94      0.78      0.85       799
        ENFP       0.97      0.83      0.90      6963
        ENTJ       0.91      0.80      0.85       531
        ENTP       0.92      0.88      0.90      4989
        ESFJ       0.65      0.89      0.75       101
        ESFP       0.71      0.87      0.78       981
        ESTJ       0.54      0.87      0.67        78
        ESTP       0.65      0.90      0.75       633
        INFJ       0.78      0.86      0.82       611
        INFP       0.85      0.90      0.87      4895
        INTJ       0.74      0.91      0.82       362
        INTP       0.92      0.82      0.86      3415
        ISFJ       0.46      0.88      0.60        75
        ISFP       0.61      0.89      0.72       670
        ISTJ       0.50      0.96      0.66        48
        ISTP       0.59      0.93      0.72     